# 10. Prepare Intellytics Source View

ODS 원천 테이블을 VOC 파이프라인이 바라보는 sandbox 작업용 view로 연결합니다.

- 원천: `kic_data_ods.intellytics_voc.intellytics_display_online_voc`
- 작업용 view: `sandbox.t_online_voc_analysis.intellytics_display_online_voc`

이 노트북을 11번 이후 파이프라인 실행 전에 먼저 실행하면, 파이프라인은 항상 ODS 원천과 동일한 최신 row를 바라봅니다.

In [ ]:
from pyspark.sql import functions as F

SOURCE_TABLE = "kic_data_ods.intellytics_voc.intellytics_display_online_voc"
TARGET_SCHEMA = "sandbox.t_online_voc_analysis"
TARGET_VIEW = "sandbox.t_online_voc_analysis.intellytics_display_online_voc"

# TARGET_VIEW 이름에 기존 managed table이 있으면 view 생성이 실패할 수 있습니다.
# True면 기존 table을 drop한 뒤 ODS 원천을 바라보는 view로 교체합니다.
REPLACE_EXISTING_TABLE_WITH_VIEW = True

print({
    "source_table": SOURCE_TABLE,
    "target_view": TARGET_VIEW,
    "replace_existing_table_with_view": REPLACE_EXISTING_TABLE_WITH_VIEW,
})


In [ ]:
# 1. 생성 전 원천/대상 상태 확인
source_cnt = spark.table(SOURCE_TABLE).count()
target_exists_before = spark.catalog.tableExists(TARGET_VIEW)

print({
    "source_cnt": source_cnt,
    "target_exists_before": target_exists_before,
})

display(spark.sql(f"DESCRIBE TABLE {SOURCE_TABLE}"))


In [ ]:
# 2. sandbox schema 준비 후 ODS 원천을 그대로 바라보는 view 생성
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

create_view_sql = f"""
CREATE OR REPLACE VIEW {TARGET_VIEW} AS
SELECT *
FROM {SOURCE_TABLE}
"""

try:
    spark.sql(create_view_sql)
except Exception as error:
    print("CREATE OR REPLACE VIEW failed. Existing object may be a managed table.")
    print(repr(error))
    if not REPLACE_EXISTING_TABLE_WITH_VIEW:
        raise

    spark.sql(f"DROP TABLE IF EXISTS {TARGET_VIEW}")
    spark.sql(create_view_sql)

print("created_view =", TARGET_VIEW)


In [ ]:
# 3. row count / post_no 기준 검증
source_df = spark.table(SOURCE_TABLE)
target_df = spark.table(TARGET_VIEW)

summary_rows = []
for name, df in [("ods_source", source_df), ("sandbox_view", target_df)]:
    item = {
        "source": name,
        "row_cnt": df.count(),
    }
    if "post_no" in df.columns:
        item["distinct_post_no_cnt"] = df.select("post_no").dropDuplicates().count()
    if "is_lifestyle" in df.columns:
        item["lifestyle_n_cnt"] = df.where(F.col("is_lifestyle") == "N").count()
        item["lifestyle_y_cnt"] = df.where(F.col("is_lifestyle") == "Y").count()
    summary_rows.append(item)

summary_df = spark.createDataFrame(summary_rows)
display(summary_df)

counts = {row["source"]: row["row_cnt"] for row in summary_df.collect()}
if counts.get("ods_source") != counts.get("sandbox_view"):
    raise ValueError(f"Row count mismatch: {counts}")

print("source view validation passed")


In [ ]:
# 4. 파이프라인 설정과 연결 확인
PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

import sys
if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

from common.config_loader import load_config, get_source_table

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")
configured_source = get_source_table(config, "raw_review_table")

print({
    "configured_raw_review_table": configured_source,
    "matches_target_view": configured_source == TARGET_VIEW,
})

if configured_source != TARGET_VIEW:
    raise ValueError(
        f"settings_intellytics.yaml raw_review_table is {configured_source}, expected {TARGET_VIEW}"
    )
